In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
import hashlib
import logging
import os
import random
import time
import traceback
from collections import defaultdict, deque
from datetime import datetime
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy import ndimage

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState


def setup_experiment_directory(base_output_dir='runs'):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = os.path.join(base_output_dir, timestamp)
    os.makedirs(base_dir, exist_ok=True)
    return base_dir, os.path.join(base_dir, 'logs.log')


def get_environment_directory(base_dir, game_id):
    d = os.path.join(base_dir, game_id)
    os.makedirs(d, exist_ok=True)
    return d


def setup_logging_for_experiment(log_file_path):
    root = logging.getLogger()
    for h in root.handlers[:]:
        if isinstance(h, logging.FileHandler):
            root.removeHandler(h); h.close()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(log_file_path, mode="w")
    fh.setLevel(root.level); fh.setFormatter(fmt)
    root.addHandler(fh)


GRID = 64
N_COLORS = 16
N_ARROW = 5
N_COORD = GRID * GRID
N_TOTAL = N_ARROW + N_COORD

ACTION_LIST = [
    GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
    GameAction.ACTION4, GameAction.ACTION5,
]
N_TIERS = 5


class FrameProcessor:
    """
    Converts raw 64x64 palette frame into:
    - status-bar-masked state hash
    - list of (action_idx, priority_tier) pairs stratified by segment
      size and color salience (5 tiers, tier-0 = highest priority)
    """

    STATUS_BAR_COLORS = {0}

    def __init__(self):
        self._cache: Dict[str, Tuple[str, List[Tuple[int, int]]]] = {}

    def _detect_active_mask(self, frame: np.ndarray) -> np.ndarray:
        mask = np.ones((GRID, GRID), dtype=bool)
        for row in range(GRID):
            if set(np.unique(frame[row])) <= self.STATUS_BAR_COLORS:
                mask[row] = False
        for row in range(GRID - 1, -1, -1):
            if set(np.unique(frame[row])) <= self.STATUS_BAR_COLORS:
                mask[row] = False
            else:
                break
        return mask

    @staticmethod
    def _tier(color: int, size: int) -> int:
        if size <= 1:   return 4
        if size <= 4:   return 3
        if size <= 16:  return 2
        if size <= 64:  return 1
        return 0

    def process(self, frame: np.ndarray) -> Tuple[str, List[Tuple[int, int]]]:
        raw_key = hashlib.md5(frame.tobytes()).hexdigest()
        if raw_key in self._cache:
            return self._cache[raw_key]

        active = self._detect_active_mask(frame)
        masked = frame.copy(); masked[~active] = 0
        state_hash = hashlib.md5(masked.tobytes()).hexdigest()

        coord_actions: List[Tuple[int, int]] = []
        for color in range(1, N_COLORS):
            color_mask = (frame == color) & active
            if not color_mask.any():
                continue
            labeled, n = ndimage.label(color_mask)
            for lbl in range(1, n + 1):
                coords = np.argwhere(labeled == lbl)
                if len(coords) == 0:
                    continue
                tier = self._tier(color, len(coords))
                cy = int(np.median(coords[:, 0]))
                cx = int(np.median(coords[:, 1]))
                action_idx = N_ARROW + cy * GRID + cx
                coord_actions.append((action_idx, tier))

        if not coord_actions:
            coord_actions = [
                (N_ARROW + y * GRID + x, 4)
                for y in range(GRID) for x in range(GRID)
                if active[y, x]
            ]

        self._cache[raw_key] = (state_hash, coord_actions)
        return state_hash, coord_actions


class NodeData:
    __slots__ = ('tier_actions', 'tested', 'transitions', 'dist_to_win', '_frame')

    def __init__(self):
        self.tier_actions: Dict[int, List[int]] = defaultdict(list)
        self.tested: Set[int] = set()
        self.transitions: Dict[int, str] = {}
        self.dist_to_win: int = 999999
        self._frame: Optional[np.ndarray] = None


class LevelGraph:
    """
    Directed state graph with:
    - Tier-grouped action lists per node
    - Explicit __RESET__ edge label (reset-loop fix from Rudakov et al. bug report)
    - BFS shortest-path to nearest frontier
    - Back-labeling of win-distances for value training (Blind Squirrel approach)
    """

    def __init__(self):
        self.nodes: Dict[str, NodeData] = {}
        self.current: Optional[str] = None
        self.pending_action: Optional[int] = None
        self.start_hash: Optional[str] = None

    def _get_or_create(self, h: str) -> NodeData:
        if h not in self.nodes:
            self.nodes[h] = NodeData()
        return self.nodes[h]

    def observe(self, state_hash: str, seg_actions: List[Tuple[int, int]],
                arrow_mask: np.ndarray, raw_frame: np.ndarray, is_reset: bool):
        node = self._get_or_create(state_hash)
        node._frame = raw_frame

        if not node.tier_actions:
            for a_idx in range(N_ARROW):
                if arrow_mask[a_idx]:
                    node.tier_actions[0].append(a_idx)
            for a_idx, tier in seg_actions:
                node.tier_actions[tier].append(a_idx)

        if self.current is not None and self.pending_action is not None:
            prev = self._get_or_create(self.current)
            prev.tested.add(self.pending_action)
            prev.transitions[self.pending_action] = '__RESET__' if is_reset else state_hash

        if self.start_hash is None:
            self.start_hash = state_hash

        self.current = state_hash
        self.pending_action = None

    def record_action(self, action_idx: int):
        self.pending_action = action_idx

    def untested(self, state_hash: str, max_tier: int) -> List[int]:
        node = self.nodes.get(state_hash)
        if node is None:
            return []
        out = []
        for t in range(max_tier + 1):
            out.extend(a for a in node.tier_actions.get(t, []) if a not in node.tested)
        return out

    def min_available_tier(self, state_hash: str) -> int:
        node = self.nodes.get(state_hash)
        if node is None:
            return N_TIERS
        for t in range(N_TIERS):
            if any(a not in node.tested for a in node.tier_actions.get(t, [])):
                return t
        return N_TIERS

    def bfs_to_frontier(self, start: str, max_tier: int) -> Optional[List[int]]:
        if self.untested(start, max_tier):
            return []
        visited = {start}
        queue: deque = deque([(start, [])])
        while queue:
            h, path = queue.popleft()
            if len(path) > 60:
                continue
            node = self.nodes.get(h)
            if node is None:
                continue
            for a_idx, dst in node.transitions.items():
                if dst == '__RESET__' or dst in visited:
                    continue
                new_path = path + [a_idx]
                if self.untested(dst, max_tier):
                    return new_path
                visited.add(dst)
                queue.append((dst, new_path))
        return None

    def back_label_win(self, win_hash: str):
        dist: Dict[str, int] = {win_hash: 0}
        q: deque = deque([win_hash])
        while q:
            h = q.popleft()
            d = dist[h]
            for src_h, node in self.nodes.items():
                for _, dst in node.transitions.items():
                    if dst == h and src_h not in dist:
                        dist[src_h] = d + 1
                        q.append(src_h)
        for h, node in self.nodes.items():
            node.dist_to_win = dist.get(h, 999999)

    def win_labeled_experiences(self) -> List[Tuple[np.ndarray, int, float]]:
        exps = []
        for h, node in self.nodes.items():
            if node._frame is None or node.dist_to_win == 999999:
                continue
            for a_idx, dst in node.transitions.items():
                if dst == '__RESET__':
                    continue
                dst_node = self.nodes.get(dst)
                if dst_node is None:
                    continue
                reward = 1.0 if dst_node.dist_to_win < node.dist_to_win else 0.0
                exps.append((node._frame, a_idx, reward))
        return exps

    def reset(self):
        self.__init__()


class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.net(x))


class ValueNet(nn.Module):
    """
    ResNet18-style network:
      input : [B, N_COLORS+1, GRID, GRID]  (colour one-hot + spatial action embedding)
      output: [B]  probability that action leads toward win
    """

    def __init__(self):
        super().__init__()
        self.action_embed = nn.Embedding(N_TOTAL, GRID * GRID)
        self.stem = nn.Sequential(
            nn.Conv2d(N_COLORS + 1, 64, 7, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.layer1 = nn.Sequential(ResBlock(64), ResBlock(64))
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128), ResBlock(128),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResBlock(256), ResBlock(256),
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            ResBlock(512), ResBlock(512),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1), nn.Sigmoid(),
        )

    def forward(self, frame_oh: torch.Tensor, action_ids: torch.Tensor) -> torch.Tensor:
        B = frame_oh.size(0)
        act_map = self.action_embed(action_ids).view(B, 1, GRID, GRID)
        x = torch.cat([frame_oh, act_map], dim=1)
        x = self.stem(x); x = self.layer1(x)
        x = self.layer2(x); x = self.layer3(x)
        x = self.layer4(x)
        return self.head(x).squeeze(1)


class MyAgent(Agent):

    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed % (2**32 - 1))
        self.start_time = time.time()

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"[HybridAgent] device={self.device}")

        self.base_dir, log_file = setup_experiment_directory()
        setup_logging_for_experiment(log_file)
        self.log_dir = get_environment_directory(self.base_dir, self.game_id)
        self.logger = logging.getLogger(f"HybridAgent_{self.game_id}")

        self.frame_proc = FrameProcessor()
        self.graph = LevelGraph()

        self.value_net = ValueNet().to(self.device)
        self.v_opt = optim.AdamW(self.value_net.parameters(), lr=1e-4, weight_decay=1e-4)
        self.v_sched = optim.lr_scheduler.CosineAnnealingLR(self.v_opt, T_max=5000, eta_min=1e-6)
        self.exp_buf: deque = deque(maxlen=100000)
        self.batch_size = 128
        self.train_freq = 4
        self.train_steps = 0

        self.current_level = -1
        self.current_tier = 0
        self.planned_path: List[int] = []
        self.prev_frame: Optional[np.ndarray] = None
        self.prev_hash: Optional[str] = None
        self.prev_action: Optional[int] = None

        self.ucb_c = 1.5
        self.epsilon = 0.03
        self.sa_counts: Dict[Tuple[str, int], int] = defaultdict(int)
        self.total_steps = 0

        self.logger.info(f"HybridAgent initialized for {self.game_id}")

    def append_frame(self, frame: FrameData) -> None:
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if frame.guid:
            self.guid = frame.guid
        if hasattr(self, "recorder") and not self.is_playback:
            import json
            self.recorder.record(json.loads(frame.model_dump_json()))

    def _level(self, f: FrameData) -> int:
        return getattr(f, 'score', None) or f.levels_completed

    def _raw_frame(self, f: FrameData) -> Optional[np.ndarray]:
        try:
            arr = np.array(f.frame, dtype=np.uint8)
            raw = arr[-1]
            assert raw.shape == (GRID, GRID)
            return raw
        except Exception:
            return None

    def _onehot(self, frame: np.ndarray) -> torch.Tensor:
        t = torch.zeros(N_COLORS, GRID, GRID, dtype=torch.float32)
        t.scatter_(0, torch.from_numpy(frame.astype(np.int64)).unsqueeze(0), 1.0)
        return t.to(self.device)

    def _arrow_mask(self, avail) -> np.ndarray:
        mask = np.zeros(N_TOTAL, dtype=bool)
        has_coord = False
        if avail:
            for a in avail:
                aid = a.value if hasattr(a, 'value') else int(a)
                if 1 <= aid <= 5:
                    mask[aid - 1] = True
                elif aid == 6:
                    has_coord = True
        else:
            mask[:N_ARROW] = True
            has_coord = True
        if has_coord:
            mask[N_ARROW:] = True
        return mask

    @torch.no_grad()
    def _value_scores(self, frame: np.ndarray, candidates: List[int]) -> np.ndarray:
        if not candidates:
            return np.array([])
        self.value_net.eval()
        oh = self._onehot(frame).unsqueeze(0).expand(len(candidates), -1, -1, -1)
        act_t = torch.tensor(candidates, dtype=torch.long, device=self.device)
        return self.value_net(oh, act_t).cpu().numpy()

    def _ucb(self, state_hash: str, candidates: List[int]) -> np.ndarray:
        log_n = np.log(max(self.total_steps, 1))
        return np.array([
            self.ucb_c * np.sqrt(log_n / max(self.sa_counts[(state_hash, a)], 1))
            for a in candidates
        ])

    def _rank(self, state_hash: str, frame: np.ndarray, candidates: List[int]) -> int:
        if random.random() < self.epsilon:
            return random.choice(candidates)
        scores = self._value_scores(frame, candidates) + self._ucb(state_hash, candidates)
        return candidates[int(np.argmax(scores))]

    def _push_exp(self, frame: np.ndarray, action: int, reward: float):
        self.exp_buf.append({'frame': frame, 'action': action, 'reward': reward})

    def _train(self):
        if len(self.exp_buf) < self.batch_size:
            return
        self.value_net.train()
        idx = np.random.choice(len(self.exp_buf), self.batch_size, replace=False)
        batch = [self.exp_buf[i] for i in idx]
        frames_oh = torch.stack([self._onehot(e['frame']) for e in batch])
        acts = torch.tensor([e['action'] for e in batch], dtype=torch.long, device=self.device)
        tgts = torch.tensor([e['reward'] for e in batch], dtype=torch.float32, device=self.device)
        loss = F.binary_cross_entropy(self.value_net(frames_oh, acts), tgts)
        self.v_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.value_net.parameters(), 1.0)
        self.v_opt.step()
        self.v_sched.step()
        self.train_steps += 1
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _back_label_and_retrain(self, win_hash: str):
        self.graph.back_label_win(win_hash)
        exps = self.graph.win_labeled_experiences()
        for frame, action, reward in exps:
            self._push_exp(frame, action, reward)
        iters = min(400, len(exps) * 2)
        for _ in range(iters):
            self._train()
        print(f"[HybridAgent] Back-labeled {len(exps)} experiences; trained {iters} steps.")

    def _reset_level(self):
        self.graph.reset()
        self.planned_path = []
        self.prev_frame = None
        self.prev_hash = None
        self.prev_action = None
        self.current_tier = 0
        self.sa_counts.clear()
        self.exp_buf.clear()
        self.value_net = ValueNet().to(self.device)
        self.v_opt = optim.AdamW(self.value_net.parameters(), lr=1e-4, weight_decay=1e-4)
        self.v_sched = optim.lr_scheduler.CosineAnnealingLR(self.v_opt, T_max=5000, eta_min=1e-6)
        print("[HybridAgent] Level reset complete.")

    def _has_time_elapsed(self) -> bool:
        return (time.time() - self.start_time) >= 8 * 3600 - 5 * 60

    def is_done(self, frames, latest_frame) -> bool:
        try:
            return latest_frame.state is GameState.WIN or self._has_time_elapsed()
        except Exception:
            return True

    def choose_action(self, frames, latest_frame):
        try:
            if self.action_counter == 0:
                print(f"[DEBUG] state={latest_frame.state} level={latest_frame.levels_completed}")
                print(f"[DEBUG] available_actions={getattr(latest_frame, 'available_actions', 'N/A')}")

            level = self._level(latest_frame)
            if level != self.current_level:
                print(f"[HybridAgent] Level {self.current_level} → {level} at step {self.action_counter}")
                self._reset_level()
                self.current_level = level

            if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                self._reset_level()
                act = GameAction.RESET
                act.reasoning = "Game reset."
                return act

            raw = self._raw_frame(latest_frame)
            if raw is None:
                act = random.choice(ACTION_LIST)
                act.reasoning = "Bad frame."
                return act

            state_hash, seg_actions = self.frame_proc.process(raw)
            avail = getattr(latest_frame, 'available_actions', None)
            arrow_mask = self._arrow_mask(avail)

            is_reset = (
                self.prev_hash is not None
                and self.graph.start_hash is not None
                and state_hash == self.graph.start_hash
                and state_hash != self.prev_hash
            )
            self.graph.observe(state_hash, seg_actions, arrow_mask, raw, is_reset)

            if self.prev_frame is not None and self.prev_action is not None:
                changed = state_hash != self.prev_hash
                self._push_exp(self.prev_frame, self.prev_action, 1.0 if changed else 0.0)

            if latest_frame.state is GameState.WIN:
                self._back_label_and_retrain(state_hash)

            self.total_steps += 1

            if self.planned_path:
                action_idx = self.planned_path.pop(0)
                if not arrow_mask[action_idx]:
                    self.planned_path = []
                else:
                    self._commit(state_hash, action_idx)
                    self.prev_frame = raw; self.prev_hash = state_hash; self.prev_action = action_idx
                    return self._to_game_action(action_idx, avail)

            untested = self.graph.untested(state_hash, self.current_tier)

            if not untested:
                path = self.graph.bfs_to_frontier(state_hash, self.current_tier)
                if path is not None:
                    if len(path) > 0:
                        self.planned_path = path[1:]
                        action_idx = path[0]
                    else:
                        untested = self.graph.untested(state_hash, self.current_tier)
                        action_idx = (self._rank(state_hash, raw, untested) if untested
                                      else self._fallback(state_hash, raw, arrow_mask))
                else:
                    self.current_tier = min(self.current_tier + 1, N_TIERS - 1)
                    untested2 = self.graph.untested(state_hash, self.current_tier)
                    action_idx = (self._rank(state_hash, raw, untested2) if untested2
                                  else self._fallback(state_hash, raw, arrow_mask))
            else:
                action_idx = self._rank(state_hash, raw, untested)

            self._commit(state_hash, action_idx)
            self.prev_frame = raw; self.prev_hash = state_hash; self.prev_action = action_idx

            if self.total_steps % self.train_freq == 0:
                self._train()

            return self._to_game_action(action_idx, avail)

        except Exception as e:
            print(f"[HybridAgent] CRASH step={self.action_counter}: {type(e).__name__}: {e}")
            traceback.print_exc()
            act = random.choice(ACTION_LIST)
            act.reasoning = f"Crash fallback: {e}"
            return act

    def _commit(self, state_hash: str, action_idx: int):
        self.graph.record_action(action_idx)
        self.sa_counts[(state_hash, action_idx)] += 1

    def _fallback(self, state_hash: str, frame: np.ndarray, mask: np.ndarray) -> int:
        valid = np.where(mask[:N_ARROW])[0].tolist() or list(range(N_ARROW))
        scores = self._value_scores(frame, valid) + self._ucb(state_hash, valid)
        return valid[int(np.argmax(scores))]

    def _to_game_action(self, action_idx: int, avail) -> GameAction:
        if action_idx < N_ARROW:
            act = ACTION_LIST[action_idx]
            act.reasoning = f"{act.name} [tier={self.current_tier}]"
            return act
        coord = action_idx - N_ARROW
        y, x = divmod(coord, GRID)
        act = GameAction.ACTION6
        act.set_data({"x": int(x), "y": int(y)})
        act.reasoning = f"ACTION6({x},{y}) [tier={self.current_tier}]"
        return act

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()